In [ ]:
!nvidia-smi


In [ ]:
!pip install -U transformers accelerate 

from transformers import AutoTokenizer, AutoModelForCausalLM #
import torch
import time

model_id = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id,dtype=torch.float16, device_map="auto")

In [ ]:
from datasets import load_dataset

math500 = load_dataset("HuggingFaceH4/MATH-500", split = "test")
print(math500)

first_example = math500[0]
print(first_example["problem"])
print(first_example["answer"])
print(first_example["level"])

In [ ]:
#detected looping manually
#function to find it automatically
def detect_loop(text, chunk_size = 20, repeat_threshold=3):
    words = text.split()#splits into individual words
    
    chunk_counts = {}#empty dictionary, key-> chunk of text, value-> how many times we have seen it

    for i in range(0, len(words)-chunk_size+1): #"last chunk that fits from the end"
        chunk_words = words[i:i+chunk_size]
        chunk = " ".join(chunk_words) #gluing words back into a sentence with a space in between each pair
 
        if chunk in chunk_counts: #checking if we have already added that chunk to the dict(is it a new chunk?)
            chunk_counts[chunk] +=1
        else:
            chunk_counts[chunk] = 1
            #updated 

    most_repeated_chunk = None
    highest_count = 0

    for chunk in chunk_counts:
        if(chunk_counts[chunk]>highest_count):
            highest_count = chunk_counts[chunk]
            most_repeated_chunk = chunk
            
    

    is_looping = highest_count>=repeat_threshold #defined a repeated threshold.

    return is_looping, highest_count, most_repeated_chunk
            


In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList

class LoopStopper(StoppingCriteria):
    def __init__(self, prompt_length, check_every=50, chunk_size=15, repeat_threshold=2):  # two new settings, with our tuned values as defaults
        self.prompt_length = prompt_length
        self.check_every = check_every
        self.chunk_size = chunk_size  # stored on self so __call__ can use it later
        self.repeat_threshold = repeat_threshold
        self.call_count = 0
        self.last_seen_length = 0

    def __call__(self, input_ids, scores, **kwargs):
        self.call_count = self.call_count + 1
        self.last_seen_length = input_ids.shape[1]

        number_generated = input_ids.shape[1] - self.prompt_length
        should_stop = False

        if number_generated > 0 and number_generated % self.check_every == 0:
            generated_tokens = input_ids[0][self.prompt_length:]
            generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            result = detect_loop(generated_text, chunk_size=self.chunk_size, repeat_threshold=self.repeat_threshold)  # pass this stopper's own settings
            should_stop = result[0]
           # print("Checked at", number_generated, "tokens -> repeat count", result[1], "-> stop:", should_stop)

        return torch.tensor([should_stop], device=input_ids.device)

In [ ]:
#picking two probs
PROBLEMS_PER_LEVEL = 10

selected_prob = []
prob_per_lev = {}

for example in math500:
    level = example["level"]
    if level not in prob_per_lev:
        prob_per_lev[level] = 0
    if prob_per_lev[level] < PROBLEMS_PER_LEVEL:  
        selected_prob.append(example) 
        prob_per_lev[level] +=1

print("picked",len(selected_prob),"problems")
print(prob_per_lev)
    

In [ ]:
#autograd - extracts answer from problem


def extract_boxed_answer(text):
    start = text.rfind(r"\boxed{")  # position of the LAST "\boxed{"; -1 if there isn't one
    if start == -1:
        return None

    position = start + len(r"\boxed{")  # jump to just after "\boxed{"
    depth = 1  # we're inside one open brace
    answer = ""  # empty string; we'll add characters to it

    while position < len(text):  # walk through the text one character at a time
        character = text[position]
        if character == "{":
            depth = depth + 1  # opened an inner brace
        if character == "}":
            depth = depth - 1  # closed a brace
        if depth == 0:
            return answer  # back to zero = the box's own closing brace: done
        answer = answer + character  # otherwise keep this character
        position = position + 1

    return None  # ran out of text before the box closed (answer got cut off)


def normalize_answer(answer):
    if answer is None:
        return None
    answer = answer.replace(" ", "")  # remove spaces
    answer = answer.replace(r"\left", "")  # \left( means the same as (
    answer = answer.replace(r"\right", "")
    answer = answer.replace(r"\dfrac", r"\frac")  # same fraction, just drawn bigger
    answer = answer.replace(r"\sqrt2", r"\sqrt{2}")

    if answer.startswith(r"\text{") and answer.endswith("}"):  # is the whole answer wrapped in \text{ ... }?
            answer = answer[len(r"\text{"):-1]  # keep only what's inside: from after "\text{" up to (not including) the last "}"
   
    return answer


print(extract_boxed_answer(r"so the answer is \boxed{\frac{1}{2}}."))  # expect: \frac{1}{2}
print(extract_boxed_answer(r"no box here"))  # expect: None
print(normalize_answer(r"\left( 3, \dfrac{\pi}{2} \right)"))  # expect: (3,\frac{\pi}{2})
    

In [ ]:
#batch run
doubt_phrases = ["wait", "double check", "double-check", "re-verify", "re-read", "let me check", "actually"]  
results = []


for prob_num in range(len(selected_prob)):
    example = selected_prob[prob_num]
    prompt = example["problem"] + "\n reason step by step, and put your final answer within \\boxed{}"
    messages = [{"role":"user","content":prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True)

    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

  #  loop_stopper = LoopStopper(prompt_length=input_ids.shape[1])
   # stopping_criteria = StoppingCriteriaList([loop_stopper])

    start_time = time.time()
    output = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens = 2000)
    end_time = time.time()

    prompt_length = input_ids.shape[1]
    full_sequence = output[0] 
    answer_tokens = full_sequence[prompt_length:]
    num_new_tok = len(answer_tokens)
    answer_text = tokenizer.decode(answer_tokens, skip_special_tokens = True)


#how did it end?

   # loop_result = detect_loop(answer_text, chunk_size=loop_stopper.chunk_size, repeat_threshold=loop_stopper.repeat_threshold)

    if num_new_tok >= 2000:
        how_it_ended = "hit cap"
    else:
        how_it_ended = "finished"
    repetition_result = detect_loop(answer_text, chunk_size=15, repeat_threshold=2)  # measured after the fact, never stops generation
    #else:
     #   how_it_ended = "finished"

#grading
    model_answer = normalize_answer(extract_boxed_answer(answer_text))
    correct_answer = normalize_answer(example["answer"])
    is_correct = model_answer == correct_answer

    text_lowercase = answer_text.lower()
    doubt_count = 0
    for phrase in doubt_phrases:
        doubt_count = doubt_count + text_lowercase.count(phrase)#checking doubt phrases
    
    one_result = {
            "problem_text": example["problem"],
            "problem_number": prob_num,
            "level": example["level"],
            "subject": example["subject"],
            "tokens": num_new_tok,
            "seconds": round(end_time - start_time, 1),
            "how_it_ended": how_it_ended,
            "model_answer": model_answer,
            "correct_answer": correct_answer,
            "is_correct": is_correct,
            "answer_text": answer_text,
            "doubt_count": doubt_count,
            "repeated_15gram": repetition_result[0],
            "top_repeat_count": repetition_result[1],
      }
    results.append(one_result)
    
    print("Problem", prob_num, "| level", example["level"], "|", how_it_ended, "|", num_new_tok, "tokens |", one_result["seconds"], "s")
    print("   model:", model_answer, "| correct:", correct_answer, "| right?", is_correct)
        


In [ ]:
#summarizing

print("per level --")

for level in [1,2,3,4,5]:
    total = 0
    correct = 0
    capped =0
    looped =0

    for one_result in results:
        if one_result["level"] == level:
            total+=1
            if one_result["is_correct"]==True:
                correct+=1
            if one_result["how_it_ended"]=="hit cap":
                capped+=1
            if one_result["how_it_ended"]=="loop":
                looped+=1
    print("Level",level,"correct", correct,"of", total, "|hit cap", capped, "|loop", looped)
print("======finished vs stuck - doubt phrases =======")

finished_count = 0
finished_doubt = 0
finished_tokens = 0
stuck_count=0
stuck_doubt = 0
stuck_tokens =0
for one_result in results:
    if one_result["how_it_ended"] == "finished":
        finished_count = finished_count + 1
        finished_doubt = finished_doubt + one_result["doubt_count"]
        finished_tokens = finished_tokens + one_result["tokens"]
    else:  # "hit cap" or "loop"
        stuck_count = stuck_count + 1
        stuck_doubt = stuck_doubt + one_result["doubt_count"]
        stuck_tokens = stuck_tokens + one_result["tokens"]

if finished_tokens > 0:  # avoid dividing by zero
    print("Finished:", finished_count, "answers | doubt per 1000 tokens:", round(finished_doubt / finished_tokens * 1000, 2))
if stuck_tokens > 0:
    print("Stuck:", stuck_count, "answers | doubt per 1000 tokens:", round(stuck_doubt / stuck_tokens * 1000, 2))



            

In [ ]:
import json

file = open("/kaggle/working/batch_results_50_nostop.json", "w")
json.dump(results, file, indent=2)
file.close()

print("Saved", len(results), "results")